# Type Annotations w Pythonie

## Spis treści:
1. Historia i cel Type Hints
2. Typy podstawowe
3. Sekwencje
4. Mapy
5. Funkcje
6. Klasy
7. Zastosowania type hints (Static vs Runtime)
8. Sprawdzanie typów (mypy, pyright)
9. Podsumowanie

---

## 1. Historia i cel Type Hints

### Kiedy się zaczęło?

| Wersja | PEP | Co dodano? |
|--------|-----|------------|
| **Python 3.0** (2008) | PEP 3107 | Function annotations - składnia, **BEZ określonego celu** |
| **Python 3.5** (2015) | PEP 484 | Type Hints - **nadano ZNACZENIE** adnotacjom |
| **Python 3.6** (2016) | PEP 526 | Variable annotations - `name: str = "Jan"` |
| **Python 3.9** (2020) | PEP 585 | Generics w built-ins - `list[int]` zamiast `List[int]` |
| **Python 3.10** (2021) | PEP 604 | Union operator - `int \| str` zamiast `Union[int, str]` |
| **Python 3.12** (2023) | PEP 695, 692, 698 | **PRZEŁOM!** Nowa składnia generics, TypedDict kwargs, @override |

### PEP 3107 (Python 3.0) - Function Annotations

**Pierwsze adnotacje** - ale Python **nie wiedział** co z nimi zrobić!

```python
# Python 3.0+ - składnia istniała, ale BEZ znaczenia
def greet(name: "imię jako string", age: "wiek jako int") -> "zwraca string":
    return f"Hi {name}, you are {age}"

# Python przechowywał to w __annotations__, ale NIC z tym nie robił
print(greet.__annotations__)
# {'name': 'imię jako string', 'age': 'wiek jako int', 'return': 'zwraca string'}
```

**To były DOWOLNE wyrażenia** - stringi, liczby, obiekty, cokolwiek. Tylko "metadane" bez celu.

### PEP 484 (Python 3.5) - Type Hints

**Nadano ZNACZENIE** adnotacjom - teraz mają konkretny cel: **typy**!

```python
# Python 3.5+ - adnotacje mają ZNACZENIE (typy)
def greet(name: str, age: int) -> str:
    return f"Hi {name}, you are {age}"

# Teraz mypy sprawdza typy!
# greet(123, "text")  # ERROR - złe typy!
```

**Od tego momentu:**
- Adnotacje = Type Hints (typy)
- Narzędzia (mypy) sprawdzają zgodność typów
- IDE używa ich do autouzupełniania

### Python 3.12 - duże zmiany w typach generycznych

**PEP 695** - nowa składnia dla generics (najprostsza składnia od zawsze):

```python
# STARY sposób (< 3.12)
from typing import TypeVar, Generic

T = TypeVar('T')

class Box(Generic[T]):
    def __init__(self, item: T) -> None:
        self.item = item

# NOWY sposób (>= 3.12) - CZYTELNIEJSZE!
class Box[T]:  # Generics bez importu!
    def __init__(self, item: T) -> None:
        self.item = item

# Funkcje generyczne
def first[T](items: list[T]) -> T:  # T zdefiniowane w składni!
    return items[0]
```

**PEP 698** - `@override` decorator (sprawdza czy metoda faktycznie override'uje):

```python
from typing import override

class Base:
    def method(self) -> None:
        pass

class Child(Base):
    @override  # mypy sprawdzi czy Base.method() istnieje!
    def method(self) -> None:
        pass
    
    @override
    def typo_method(self) -> None:  # ERROR - Base nie ma tej metody!
        pass
```

### Type hints = w ostatnich latach jedna z NAJBARDZIEJ rozwijanych funkcji Pythona!

**Od Python 3.5 (2015) do dziś:**
- **W KAŻDYM release'ie** nowe PEPy dla type hints
- Często **niemałe zmiany** (nowa składnia, nowe typy, nowe możliwości)
- **Obecnie** (2024-2026) dalej intensywnie rozwijane

**Dlaczego tak intensywny rozwój?**
1. **FastAPI, Pydantic, SQLAlchemy** - duże frameworki używają type hints
2. **Type checkers** (mypy, pyright) stają się standardem w projektach
3. **IDE** (VS Code, PyCharm) mocno bazują na type hints
4. Python stara się **nie pozostać w tyle** za językami statycznie typowanymi

**Przykłady zmian w kolejnych wersjach:**
- 3.5 → Type hints foundation
- 3.6 → Variable annotations
- 3.7 → `from __future__ import annotations` (postponed evaluation)
- 3.8 → Literal, TypedDict, Final
- 3.9 → Generics w built-ins (`list[int]`)
- 3.10 → Union operator (`|`)
- 3.11 → Self type, TypeVarTuple
- 3.12 → **Nowa składnia generics**
- 3.13+ → Dalszy rozwój...

**Wniosek:** Type hints to **żywa, rozwijająca się funkcja**. Warto śledzić nowe release'y Pythona!

### Po co?

Python jest **dynamicznie typowany** - zmienne mogą zmieniać typ w runtime:

```python
x = 5        # int
x = "hello"  # teraz str - nie ma błędu!
```

**Problemy:**
- ❌ Błędy typów wykrywane dopiero w runtime
- ❌ Trudno zrozumieć API funkcji bez czytania kodu
- ❌ Brak autouzupełniania w IDE

**Type Hints rozwiązują to:**
- ✅ Błędy typów wykrywane PRZED uruchomieniem (mypy, pyright)
- ✅ Dokumentacja "w kodzie"
- ✅ Autouzupełnianie i podpowiedzi w IDE
- ✅ **FastAPI używa type hints do generowania API, walidacji, dokumentacji!**

**WAŻNE:** Type hints **NIE WPŁYWAJĄ** na wykonanie kodu - Python je **ignoruje** w runtime!

```python
def add(a: int, b: int) -> int:
    return a + b

# Python pozwoli na to (runtime nie sprawdza!):
add("hello", "world")  # Zwróci "helloworld" bez błędu
```

Sprawdzanie typów = **narzędzia zewnętrzne** (mypy, pyright) lub **biblioteki** (Pydantic, FastAPI).

---

## 2. Typy podstawowe

### Built-in types

Podstawowe typy - bez importu:

In [ ]:
# Podstawowe typy
age: int = 25
name: str = "Jan"
price: float = 19.99
is_active: bool = True
nothing: None = None

print(f"age={age}, name={name}, price={price}, is_active={is_active}, nothing={nothing}")

### Union - wiele typów

Zmienna może mieć **jeden z kilku typów**:

In [ ]:
from typing import Union

# Stary sposób (Python < 3.10)
user_id: Union[int, str] = 123  # może być int LUB str
user_id = "abc"  # OK

# Nowy sposób (Python >= 3.10) - LEPSZY!
user_id: int | str = 456
user_id = "def"  # OK

print(user_id)

### Optional - wartość lub None

`Optional[T]` = `T | None` = wartość może być typu T **lub** None:

In [ ]:
from typing import Optional

# Stary sposób
middle_name: Optional[str] = None  # str lub None
middle_name = "Kowalski"  # OK

# Nowy sposób (Python >= 3.10) - LEPSZY!
middle_name: str | None = None
middle_name = "Nowak"  # OK

print(middle_name)

### Any - dowolny typ

"Wyłącz sprawdzanie typów dla tej zmiennej":

In [ ]:
from typing import Any

# Może być WSZYSTKO
data: Any = 42
data = "text"
data = [1, 2, 3]
data = {"key": "value"}

print(data)

# Używaj TYLKO gdy naprawdę nie wiesz jaki będzie typ
# (np. JSON z API, dict z unknown structure)

### Literal - konkretna wartość

Zmienna może mieć tylko **jedną z wymienionych wartości**:

In [ ]:
from typing import Literal

# Status może być TYLKO "pending", "active" lub "inactive"
status: Literal["pending", "active", "inactive"] = "pending"
status = "active"  # OK
# status = "invalid"  # mypy/pyright ERROR!

print(status)

# Przydatne w FastAPI dla enumeracji wartości!

### TypeAlias - alias typu (Python >= 3.10)

Nadaj nazwę skomplikowanemu typowi:

In [ ]:
from typing import TypeAlias

# Zamiast powtarzać Union[int, str] wszędzie
UserId: TypeAlias = int | str

# Teraz używaj UserId
def get_user(user_id: UserId) -> dict:
    return {"id": user_id}

print(get_user(123))
print(get_user("abc"))

### NewType - rozróżnianie aliasów typów

TypeAlias to tylko **alias** - mypy traktuje `UserId` i `int` identycznie.

NewType tworzy **"nowy typ"** - mypy NIE pozwoli mieszać `UserId` z `int`:

In [ ]:
from typing import NewType

# UserId i ArticleId to "różne typy" (dla mypy, w runtime to int)
UserId = NewType('UserId', int)
ArticleId = NewType('ArticleId', int)

def get_user(user_id: UserId) -> str:
    return f"User {user_id}"

def get_article(article_id: ArticleId) -> str:
    return f"Article {article_id}"

# Musisz "oznaczyć" wartość jako UserId/ArticleId
user_id = UserId(123)
article_id = ArticleId(456)

print(get_user(user_id))        # ✅ OK
# print(get_user(article_id))   # ❌ mypy ERROR! Różne typy
# print(get_user(123))          # ❌ mypy ERROR! int ≠ UserId

print(f"\nW runtime: type(user_id) = {type(user_id).__name__} (zwykły int)")
print("W type checkerze: mypy traktuje UserId i int jako RÓŻNE typy")

---

## 3. Sekwencje

### list[T] - lista elementów typu T

In [ ]:
# Python >= 3.9 - używaj list[T] (bez importu!)
numbers: list[int] = [1, 2, 3, 4, 5]
names: list[str] = ["Jan", "Anna", "Piotr"]

# Lista różnych typów (rzadko używane)
mixed: list[int | str] = [1, "two", 3, "four"]

print(numbers, names, mixed)

In [ ]:
# Python < 3.9 - import z typing (STARY SPOSÓB)
from typing import List

numbers: List[int] = [1, 2, 3]  # Zamiast list[int]

### tuple[T, ...] - krotka

Dwa sposoby:
1. **Tuple o stałej długości** - wymień typy wszystkich elementów
2. **Tuple dowolnej długości** - użyj `tuple[T, ...]`

In [ ]:
# Tuple o STAŁEJ długości (3 elementy)
person: tuple[str, int, bool] = ("Jan", 25, True)
# person[0] -> str
# person[1] -> int
# person[2] -> bool

# Tuple DOWOLNEJ długości (wszystkie int)
coordinates: tuple[int, ...] = (1, 2, 3, 4, 5)  # może mieć dowolną liczbę elementów

print(person, coordinates)

### set[T] - zbiór unikalnych elementów

In [ ]:
# Zbiór unikalnych ID
user_ids: set[int] = {1, 2, 3, 4, 5}
tags: set[str] = {"python", "fastapi", "web"}

print(user_ids, tags)

### Sequence - abstrakcyjna sekwencja (read-only)

Funkcja przyjmuje **dowolną sekwencję** (list, tuple, str, range, ...):

In [ ]:
from typing import Sequence

# Funkcja przyjmuje list, tuple, str, range, ...
def print_items(items: Sequence[int]) -> None:
    for item in items:
        print(item, end=" ")
    print()

print_items([1, 2, 3])        # list - OK
print_items((4, 5, 6))        # tuple - OK
print_items(range(7, 10))     # range - OK

# Sequence = "czytaj, ale nie modyfikuj"
# Jeśli funkcja MA MODYFIKOWAĆ -> użyj list[T] lub MutableSequence[T]

### Iterable - coś co można iterować

In [ ]:
from typing import Iterable

# Przyjmuje WSZYSTKO co można iterować (list, tuple, set, dict, str, generator, ...)
def sum_numbers(numbers: Iterable[int]) -> int:
    return sum(numbers)

print(sum_numbers([1, 2, 3]))       # list
print(sum_numbers((4, 5, 6)))       # tuple
print(sum_numbers({7, 8, 9}))       # set
print(sum_numbers(range(10, 13)))   # range

---

## 4. Mapy (słowniki)

### dict[K, V] - słownik

In [ ]:
# Python >= 3.9
user: dict[str, int] = {"age": 25, "score": 100}
prices: dict[str, float] = {"apple": 2.5, "banana": 1.2}

# Klucze i wartości różnych typów
mixed: dict[str, int | str] = {"age": 25, "name": "Jan"}

print(user, prices, mixed)

### TypedDict - słownik o stałej strukturze

Zwykły `dict[str, Any]` nie mówi **jakie klucze** są w słowniku.  
`TypedDict` definiuje **strukturę** słownika:

In [ ]:
from typing import TypedDict

# Definiujemy strukturę
class User(TypedDict):
    id: int
    name: str
    email: str
    is_active: bool

# Używamy
user: User = {
    "id": 1,
    "name": "Jan",
    "email": "jan@example.com",
    "is_active": True
}

print(user)
print(user["name"])  # IDE podpowie dostępne klucze!

# mypy sprawdzi czy słownik ma wszystkie wymagane klucze!

**Opcjonalne klucze** - użyj `NotRequired` (Python >= 3.11) lub `total=False`:

In [ ]:
from typing import TypedDict, NotRequired

class User(TypedDict):
    id: int
    name: str
    email: NotRequired[str]  # Opcjonalny!

user: User = {"id": 1, "name": "Jan"}  # OK - email nie jest wymagany
print(user)

---

## 5. Funkcje

### Podstawowe adnotacje funkcji

In [ ]:
# Parametry i typ zwracany
def add(a: int, b: int) -> int:
    return a + b

# Funkcja nie zwraca wartości
def print_message(msg: str) -> None:
    print(msg)

# Domyślne wartości parametrów
def greet(name: str, greeting: str = "Hello") -> str:
    return f"{greeting}, {name}!"

print(add(2, 3))
print_message("Hi")
print(greet("Jan"))
print(greet("Anna", "Hi"))

### *args i **kwargs z typami

In [ ]:
# *args - dowolna liczba argumentów pozycyjnych
def sum_all(*numbers: int) -> int:
    return sum(numbers)

print(sum_all(1, 2, 3, 4, 5))

# **kwargs - dowolna liczba argumentów nazwanych
def print_kwargs(**kwargs: str) -> None:
    for key, value in kwargs.items():
        print(f"{key}={value}")

print_kwargs(name="Jan", city="Warsaw")

# Razem
def flexible(a: int, *args: int, **kwargs: str) -> None:
    print(f"a={a}, args={args}, kwargs={kwargs}")

flexible(1, 2, 3, name="Jan", city="Warsaw")

### Callable - typ dla funkcji

Parametr lub zmienna przechowuje **funkcję**:

In [ ]:
from typing import Callable

# Callable[[arg1_type, arg2_type], return_type]
# Funkcja przyjmuje (int, int) i zwraca int
def apply_operation(a: int, b: int, operation: Callable[[int, int], int]) -> int:
    return operation(a, b)

def add(x: int, y: int) -> int:
    return x + y

def multiply(x: int, y: int) -> int:
    return x * y

print(apply_operation(5, 3, add))       # 8
print(apply_operation(5, 3, multiply))  # 15

### @overload - wiele sygnatur funkcji

Funkcja zachowuje się **inaczej** w zależności od typów argumentów:

In [ ]:
from typing import overload

# Deklaracje overload (NIE IMPLEMENTACJA!)
@overload
def process(x: int) -> str: ...

@overload
def process(x: str) -> int: ...

# Rzeczywista implementacja
def process(x: int | str) -> int | str:
    if isinstance(x, int):
        return f"Number: {x}"
    else:
        return len(x)

# mypy sprawdzi:
result1: str = process(42)      # OK - int -> str
result2: int = process("hello") # OK - str -> int
print(result1)
print(result2)

---

## 6. Klasy

### Atrybuty klasowe i instancyjne

In [ ]:
class User:
    # Atrybut klasowy
    total_users: int = 0
    
    # Atrybuty instancyjne - adnotacja w __init__
    def __init__(self, name: str, age: int) -> None:
        self.name: str = name  # można pominąć typ (wynika z parametru)
        self.age = age         # to też OK
        User.total_users += 1
    
    def greet(self) -> str:        return f"Hi, I'm {self.name}, {self.age} years old"

user = User("Jan", 25)
print(user.greet())
print(f"Total users: {User.total_users}")

### Metody - self i cls

In [ ]:
class MyClass:
    count: int = 0
    
    # Metoda instancyjna - self (NIE ADNOTUJ self!)
    def instance_method(self, x: int) -> int:
        return x * 2
    
    # Metoda klasowa - cls (NIE ADNOTUJ cls!)
    @classmethod
    def class_method(cls, x: int) -> int:
        cls.count += x
        return cls.count
    
    # Metoda statyczna - bez self/cls
    @staticmethod
    def static_method(x: int, y: int) -> int:
        return x + y

obj = MyClass()
print(obj.instance_method(5))
print(MyClass.class_method(10))
print(MyClass.static_method(3, 7))

### Generic[T] - klasa generyczna

**Czym jest generyk?**

Klasa/funkcja która działa z **parametrem typu** - "nie wiem jeszcze jaki to będzie typ, ale jak już wstawisz int, to wszędzie w tej instancji T musi być int".

**T to jak "zmienna" dla typu** - placeholder który podstawiasz przy tworzeniu instancji.

**Po co?**

### 1. Type safety - spójność typu w całej klasie

```python
class Box[T]:  # T = parametr typu (będzie podstawiony później), scope klasy w której został zdefiniowany (tutaj Box)
    def __init__(self, item: T):
        self.item = item
    
    def get(self) -> T:  # Zwraca TEN SAM typ co wstawiony w __init__
        return self.item

# Dla int_box: T = int (wszędzie gdzie T, tam int)
int_box = Box[int](42)
result: int = int_box.get()  # ✅ mypy wie że zwraca int

# Dla str_box: T = str (wszędzie gdzie T, tam str)
str_box = Box[str]("hello")
result: str = str_box.get()  # ✅ mypy wie że zwraca str

# BŁĄD - mieszasz typy!
int_box = Box[int](42)
wrong: str = int_box.get()  # ❌ mypy ERROR! Zwraca int, nie str
```

**Istota:** "Jak wstawisz int, to wszędzie T = int (dla tej zmiennej)". Każda instancja ma swoje T, ale w jej obrębie jest spójne.

### 2. IDE wie co zwracasz

```python
int_box = Box[int](42)
result = int_box.get()  # IDE wie że result: int
                        # Podpowie metody dla int (.bit_length(), ...)

str_box = Box[str]("hello")
result = str_box.get()  # IDE wie że result: str
                        # Podpowie metody dla str (.upper(), .lower(), ...)
```

### 3. Jedna klasa zamiast wielu

```python
# ❌ BEZ GENERICS - jeżeli chciałbyś uzyskać ten sam efekt w mypy musisz tworzyć osobne klasy dla każdego typu (gdybyś spróbował unii,
# np. int|float|str to stracisz informacje o konkretnym typie, wiesz tylko, że to powinno być coś z worka typów
class IntBox:
    def __init__(self, item: int): ...
    def get(self) -> int: ...
    
class StrBox:
    def __init__(self, item: str): ...
    def get(self) -> str: ...
    
class FloatBox:
    def __init__(self, item: float): ...
    def get(self) -> float: ...

# ✅ Z GENERICS - jedna klasa dla wszystkich typów
class Box[T]:
    def __init__(self, item: T): ...
    def get(self) -> T: ...
```

**Podsumowanie:**
- Generyk ≠ "tworzysz nowy typ"
- Generyk = "klasa z placeholderem dla typu"
- T podstawiasz przy tworzeniu instancji (Box[int], Box[str])
- W obrębie jednej instancji T jest spójne (jak int, to wszędzie int)

**Przykład (Python >= 3.12 - nowa składnia):**

In [ ]:
# Python >= 3.12 - nowa składnia (NAJLEPSZA!)
class Box[T]:
    def __init__(self, item: T) -> None:
        self.item = item
    
    def get(self) -> T:
        return self.item

int_box: Box[int] = Box(42)
str_box: Box[str] = Box("hello")

print(int_box.get())  # 42 (IDE wie że int)
print(str_box.get())  # "hello" (IDE wie że str)

# Python < 3.12 - stara składnia (wymaga importu)
from typing import Generic, TypeVar

T = TypeVar('T')

class OldBox(Generic[T]):
    def __init__(self, item: T) -> None:
        self.item = item
    
    def get(self) -> T:
        return self.item

# Używanie takie samo
old_int_box: OldBox[int] = OldBox(42)
print(old_int_box.get())

---

## 7. Zastosowania type hints

Type hints są używane **dwojako**:
1. **Static** - tylko type checker (mypy, pyright) - sprawdzanie przed uruchomieniem
2. **Runtime** - niektóre biblioteki czytają type hints podczas wykonania programu

### 7.1 Static - tylko type checker

**Python runtime IGNORUJE te type hints!**  
Używane TYLKO przez mypy/pyright do sprawdzania typów.

#### Protocol - structural subtyping ("duck typing")

Sprawdza czy klasa **ma odpowiednie metody**, nie czy **dziedziczy** z konkretnej klasy:

In [ ]:
from typing import Protocol

# Protokół - "ma metodę draw()"
class Drawable(Protocol):
    def draw(self) -> str:
        ...

# Klasy NIE dziedziczą z Drawable, ale MAJĄ metodę draw()
class Circle:
    def draw(self) -> str:
        return "Drawing circle"

class Square:
    def draw(self) -> str:
        return "Drawing square"

# Funkcja przyjmuje WSZYSTKO co ma draw()
def render(shape: Drawable) -> None:
    print(shape.draw())

render(Circle())  # OK
render(Square())  # OK
# render("string")  # ERROR - str nie ma draw()

print("Protocol demo complete")

**Python runtime:**
- ✅ `render(Circle())` zadziała - Circle ma draw()
- ✅ `render("string")` też "zadziała" (tzn. błąd wystąpi dopiero podczas wykonania "string".draw(), Python nie sprawdza Protocol!)
- ❌ Dopiero mypy/pyright zgłosi błąd

**Protocol to TYLKO dla type checkera!**

### 7.2 Runtime - biblioteki używają type hints

**Te biblioteki CZYTAJĄ type hints w runtime** i robią z nimi coś (walidacja, generowanie kodu, etc.)

#### 1. dataclass - generowanie kodu

`@dataclass` czyta `__annotations__` i **generuje** `__init__`, `__repr__`, `__eq__`:

In [ ]:
from dataclasses import dataclass

@dataclass
class User:
    id: int
    name: str
    email: str
    is_active: bool = True

# __init__, __repr__, __eq__ wygenerowane automatycznie!
user = User(id=1, name="Jan", email="jan@example.com")
print(user)

user2 = User(1, "Jan", "jan@example.com")
print(f"user == user2: {user == user2}")

# ⚠️ dataclass NIE waliduje typów!
bad_user = User(id="not_int", name=123, email=[])
print(f"Bad user (Python pozwala!): {bad_user}")

#### 2. Pydantic - walidacja typów w runtime

**Pydantic czyta type hints i WALIDUJE dane!**

In [ ]:
from pydantic import BaseModel, Field
from typing import Annotated

class User(BaseModel):
    id: int
    name: Annotated[str, Field(min_length=3, max_length=50)]
    age: Annotated[int, Field(ge=0, le=120)]
    email: str

# Pydantic waliduje:
user = User(id=1, name="Jan", age=25, email="jan@example.com")
print(user)

# Annotated[typ, metadane] - dodaje walidację do typu
print(f"\nAnnotated = typ + metadane (Field z regułami walidacji)")

# Odkomentuj żeby zobaczyć ValidationError:
# bad_user = User(id="bad", name="Jan", age=25, email="jan@example.com")  # ERROR!
# bad_user = User(id=1, name="Jo", age=25, email="jan@example.com")  # ERROR - za krótkie!

#### 3. FastAPI - walidacja requestów + dokumentacja

**FastAPI używa type hints do:**
- Walidacji requestów (przez Pydantic)
- Generowania dokumentacji OpenAPI
- Serializacji/deserializacji JSON

```python
from fastapi import FastAPI, Query
from typing import Annotated

app = FastAPI()

@app.get("/items/")
def read_items(
    q: Annotated[str | None, Query(min_length=3, max_length=50)] = None,
    skip: Annotated[int, Query(ge=0)] = 0,
    limit: Annotated[int, Query(ge=1, le=100)] = 10
):
    return {"q": q, "skip": skip, "limit": limit}

# FastAPI automatycznie:
# - Waliduje że skip >= 0
# - Waliduje że limit między 1 a 100
# - Generuje OpenAPI docs z tymi ograniczeniami
```

#### 4. SQLAlchemy 2.0 - Mapped[T]

**SQLAlchemy czyta type hints i generuje kolumny bazy danych:**

In [ ]:
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users_demo"
    
    # Mapped[typ] - kolumna w bazie danych
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column()
    email: Mapped[str] = mapped_column(unique=True)
    age: Mapped[int | None] = mapped_column(default=None)  # Nullable

# SQLAlchemy:
# - Generuje kolumny: id INTEGER, name VARCHAR, email VARCHAR UNIQUE, age INTEGER NULL
# - IDE wie że user.id to int, user.name to str
# - mypy sprawdzi typy!

print("Model User created (Mapped[T] używane)")

**Różnica vs stary styl (SQLAlchemy 1.x):**

```python
# ❌ BEZ Mapped - brak type hints
class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)  # IDE nie wie jaki typ!

# ✅ Z Mapped - type safety
class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)  # IDE wie że int!
```

### Podsumowanie zastosowań

| Zastosowanie | Static/Runtime | Co robi? |
|--------------|----------------|----------|
| **Protocol** | Static | Duck typing dla type checkera |
| **dataclass** | Runtime | Generuje `__init__`, `__repr__`, `__eq__` (NIE waliduje!) |
| **Pydantic** | Runtime | **Waliduje typy** w runtime |
| **FastAPI** | Runtime | Walidacja + generowanie OpenAPI docs |
| **SQLAlchemy Mapped** | Runtime | Generuje kolumny bazy danych |

**Wniosek:** Type hints to nie tylko mypy! Wiele bibliotek używa ich w runtime.

---

## 8. Sprawdzanie typów

Type hints **NIE SĄ SPRAWDZANE** przez Python w runtime!  
Musisz użyć **zewnętrznych narzędzi**.

### mypy - najpopularniejszy type checker

In [ ]:
# Instalacja
# pip install mypy

# Użycie
# mypy app.py
# mypy .

# Przykład pliku app.py:
def add(a: int, b: int) -> int:
    return a + b

result: str = add(2, 3)  # ERROR! add() zwraca int, nie str

# mypy wykryje błąd:
# app.py:4: error: Incompatible types in assignment (expression has type "int", variable has type "str")

### pyright / pylance - type checker w VS Code

**Pylance** (extension w VS Code) używa **pyright** pod spodem.

```bash
# Instalacja pyright (opcjonalnie - Pylance ma go wbudowanego)
pip install pyright

# Użycie
pyright app.py
pyright .
```

**VS Code + Pylance:**
- Automatyczne sprawdzanie typów podczas pisania
- Podkreśla błędy typów na czerwono
- Autouzupełnianie na podstawie typów

### ruff - linter + formatter

```bash
pip install ruff

# Sprawdzanie
ruff check .

# Formatowanie
ruff format .
```

Ruff **NIE JEST** type checkerem (jak mypy/pyright), ale sprawdza **style** i **potencjalne błędy**.

### Konfiguracja mypy (pyproject.toml)

```toml
[tool.mypy]
python_version = "3.12"
strict = true  # Ścisłe sprawdzanie
warn_return_any = true
warn_unused_configs = true
disallow_untyped_defs = true  # Wszystkie funkcje MUSZĄ mieć typy
```

### Automatyczne dodawanie type hints

Masz stary kod bez type hints? Nie musisz typować ręcznie! Są narzędzia które **analizują działający kod** i **automatycznie dodają typy**.

#### monkeytype (Instagram)

```bash
pip install monkeytype

# 1. Uruchom program z monitoringiem
monkeytype run script.py

# 2. Zobacz co wykrył
monkeytype stub module_name

# 3. Dodaj typy do pliku
monkeytype apply module_name
```

#### pyannotate (twórcy mypy)

```bash
pip install pyannotate

# 1. Uruchom z monitoringiem
pyannotate --type-info type_info.json script.py

# 2. Wygeneruj adnotacje
pyannotate --type-info type_info.json
```

**Ważne:** Te narzędzia analizują **runtime behavior**, więc typy mogą być niepełne (zależą od wywołań podczas uruchomienia).

### Exit codes i CI/CD

**mypy zwraca exit code:**
- `0` - wszystko OK (brak błędów typów)
- `1` - znaleziono błędy typów

**Sprawdzenie exit code:**

```bash
# Unix/Linux/macOS (bash)
mypy script.py
echo $?  # Wyświetli 0 lub 1

# Windows CMD
mypy script.py
echo %errorlevel%  # Wyświetli 0 lub 1

# Windows PowerShell
mypy script.py
echo $?  # Wyświetli True (sukces) lub False (błąd)
```

**Dlaczego to ważne? CI/CD!**

Systemy CI/CD (Jenkins, GitHub Actions, GitLab CI) działają prosto:
1. Uruchamiają listę poleceń shellowych
2. Jeśli jakieś polecenie zakończy się exit code ≠ 0 → **build FAILED** ❌
3. Jeśli wszystkie zakończą się exit code = 0 → **build SUCCESS** ✅

**Przykład GitHub Actions:**

```yaml
# .github/workflows/ci.yml
name: CI
on: [push]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install dependencies
        run: pip install mypy
      - name: Type check
        run: mypy .  # Jeśli błędy typów → exit 1 → build failed!
```

**Wniosek:** Dodajesz type hints + mypy do CI/CD → automatyczna weryfikacja typów przy każdym commicie. Nic nie tracisz (type hints nie zmieniają działania kodu), a zyskujesz dodatkowe sprawdzenie!

---

## 9. Podsumowanie

### Najważniejsze punkty:

1. **Type hints NIE WPŁYWAJĄ na runtime** - Python je ignoruje, sprawdzają je zewnętrzne narzędzia (mypy, pyright)

2. **FastAPI UŻYWA type hints**:
   - Walidacja requestów
   - Generowanie dokumentacji (OpenAPI)
   - Serializacja/deserializacja

3. **Pydantic UŻYWA type hints** do walidacji danych w runtime

4. **SQLAlchemy 2.0 UŻYWA `Mapped[T]`** dla type safety

### Co używać? (Python >= 3.10)

| Stary sposób (< 3.9/3.10) | Nowy sposób (>= 3.9/3.10) | Notatka |
|---------------------------|---------------------------|----------|
| `List[int]` | `list[int]` | Bez importu z typing |
| `Dict[str, int]` | `dict[str, int]` | Bez importu |
| `Tuple[int, str]` | `tuple[int, str]` | Bez importu |
| `Set[str]` | `set[str]` | Bez importu |
| `Union[int, str]` | `int \| str` | Czytelniejsze |
| `Optional[str]` | `str \| None` | Jawniejsze |

### Sprawdzanie typów - narzędzia:

```bash
# Instalacja
pip install mypy pyright ruff

# Użycie
mypy .           # Type checking
pyright .        # Type checking (szybszy)
ruff check .     # Linting
ruff format .    # Formatting
```

### Kiedy używać jakich typów?

| Typ | Kiedy używać? |
|-----|---------------|
| `int`, `str`, `float`, `bool` | Podstawowe typy |
| `list[T]`, `dict[K, V]` | Kolekcje (znasz typ elementów) |
| `Any` | Nie wiesz jaki będzie typ (rzadko!) |
| `T \| None` | Wartość opcjonalna |
| `Literal["a", "b"]` | Enum wartości |
| `TypedDict` | Słownik o stałej strukturze |
| `Callable` | Parametr/zmienna to funkcja |
| `Protocol` | Duck typing ("ma metodę X") |
| `Generic[T]` | Klasa generyczna (dla wielu typów) |
| `Annotated[T, ...]` | Typ + metadane (Pydantic, FastAPI) |
| `Mapped[T]` | SQLAlchemy 2.0 kolumny |

### Następny krok:

Ćwiczenie - **popraw wszystkie błędy typów** w kodzie! 🚀